# 05I-B — Held-versus-Burned-Over Input Regimes

This notebook characterizes small, interpretable combinations of frozen-model inputs associated with the two fireline outcomes:

- `hold_label = 1`: **Held**
- `hold_label = 0`: **Burned Over**

The primary analysis uses only physical/environmental predictors (fire approach, weather, terrain, and fuels/canopy). A separate sensitivity analysis permits the 11 observation-process variables.

Scientific guardrails:

- The frozen 05C primary model is **not refit or modified**.
- All surrogate targets are the saved incident-held-out OOF probabilities from 05C.
- Surrogate fidelity is evaluated across the same incident-disjoint outer folds.
- Final rule thresholds are descriptive summaries, not universal operational thresholds.
- Regimes are predictive associations, not causal effects.


## 0. Configuration and frozen input contract


In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "05IB_V1_HELD_VS_BURNED_OVER_INPUT_REGIMES"

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))
UNIT_DESIGN_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_design_v1" / "unit_outer_fold_and_training_weights_v1.parquet"
MASTER_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_table_assembly_v1" / "fireline_hold_modeling_master_v1.parquet"
PRIMARY_OOF_PATH = PROJECT_ROOT / "data" / "processed" / "baseline_modeling_v1" / "oof_predictions" / "E_FULL_EXTENDED__row.parquet"
FEATURE_IMPORTANCE_PATH = PROJECT_ROOT / "outputs" / "frozen_model_interpretation_v1" / "OOF_SHAP_FEATURE_IMPORTANCE_V1.csv"
FEATURE_GROUPS_PATH = PROJECT_ROOT / "outputs" / "frozen_model_interpretation_v1" / "INTERPRETATION_FEATURE_GROUPS_V1.csv"
PROXY_FEATURES_PATH = PROJECT_ROOT / "outputs" / "observation_process_proxy_sensitivity_v1" / "OBSERVATION_PROCESS_PROXY_FEATURES_V1.csv"
INTERPRETATION_CHECKS_PATH = PROJECT_ROOT / "outputs" / "frozen_model_interpretation_v1" / "FINAL_FROZEN_MODEL_INTERPRETATION_CHECKS_V1.csv"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "input_regimes_v1"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "input_regimes_v1"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
for p in [OUTPUT_ROOT, PROCESSED_ROOT, FIGURE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

for p in [UNIT_DESIGN_PATH, MASTER_PATH, PRIMARY_OOF_PATH, FEATURE_IMPORTANCE_PATH, FEATURE_GROUPS_PATH, PROXY_FEATURES_PATH, INTERPRETATION_CHECKS_PATH]:
    assert p.exists(), p

EXPECTED_UNITS = 304_143
EXPECTED_INCIDENTS = 381
OUTER_FOLDS = 5
RANDOM_SEED = 20260826
MAX_TREE_DEPTH = 3
MIN_SAMPLES_LEAF = 2_500
MIN_WEIGHT_FRACTION_LEAF = 0.025
MAX_PHYSICAL_FEATURES = 24
MIN_STABLE_FEATURES = 8
MIN_TOP20_FOLD_COUNT = 3
MIN_FEATURES_PER_PHYSICAL_SOURCE = 3
MISSING_INDICATOR_THRESHOLD = 0.005
BOOTSTRAP_REPLICATES = 2_000
MIN_REGIME_INCIDENTS = 20

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
}
for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

from datetime import datetime, timezone
import textwrap

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor, plot_tree
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 420)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 240, "axes.spines.top": False, "axes.spines.right": False})

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)


## 1. Select stable physical predictors from the frozen 05I interpretation


In [ ]:
importance = pd.read_csv(FEATURE_IMPORTANCE_PATH).sort_values("rank_oof_shap")
groups = pd.read_csv(FEATURE_GROUPS_PATH)
proxy_features = pd.read_csv(PROXY_FEATURES_PATH)["excluded_proxy_feature"].astype(str).tolist()
proxy_set = set(proxy_features)
interpretation_checks = pd.read_csv(INTERPRETATION_CHECKS_PATH)

assert len(importance) == 803 and importance["feature"].is_unique
assert len(groups) == 803 and groups["feature"].is_unique
assert len(proxy_features) == 11 and len(proxy_set) == 11
assert interpretation_checks["pass"].astype(bool).all()

physical_pool = importance[
    (importance["source_group"] != "observation_process")
    & (importance["top_20_fold_count"] >= MIN_TOP20_FOLD_COUNT)
].copy()

if len(physical_pool) < MIN_STABLE_FEATURES:
    physical_pool = importance[importance["source_group"] != "observation_process"].copy()

physical_sources = ["weather", "fuels_canopy", "terrain", "fire_approach_physical"]
physical_candidates = physical_pool["feature"].tolist()
for source_group in physical_sources:
    source_top = importance[importance["source_group"] == source_group].head(MIN_FEATURES_PER_PHYSICAL_SOURCE)["feature"].tolist()
    physical_candidates.extend(source_top)
physical_candidates = list(dict.fromkeys(physical_candidates))
physical_candidates = (
    importance[importance["feature"].isin(physical_candidates)]
    .sort_values("rank_oof_shap")
    .head(MAX_PHYSICAL_FEATURES)["feature"]
    .tolist()
)
all_input_candidates = physical_candidates + [f for f in proxy_features if f not in physical_candidates]

assert len(physical_candidates) >= MIN_STABLE_FEATURES
assert not (set(physical_candidates) & proxy_set)
assert proxy_set.issubset(set(all_input_candidates))
assert set(physical_sources) == set(importance.set_index("feature").loc[physical_candidates, "source_group"])

candidate_contract = importance[importance["feature"].isin(all_input_candidates)].copy()
candidate_contract["analysis_role"] = np.where(
    candidate_contract["feature"].isin(proxy_set),
    "observation_process_sensitivity_only",
    "primary_physical_regime_candidate",
)
candidate_contract["sensitivity_missing_indicator_created"] = candidate_contract["sample_missing_fraction"] >= MISSING_INDICATOR_THRESHOLD
candidate_contract = candidate_contract.sort_values(["analysis_role", "rank_oof_shap"])
candidate_contract.to_csv(OUTPUT_ROOT / "INPUT_REGIME_FEATURE_CONTRACT_V1.csv", index=False)

display(candidate_contract[[
    "analysis_role", "rank_oof_shap", "feature", "source_group", "feature_family",
    "top_20_fold_count", "sample_missing_fraction", "sensitivity_missing_indicator_created",
]])
print("Physical candidates:", len(physical_candidates))
print("Sensitivity candidates:", len(all_input_candidates))


## 2. Load the saved OOF predictions and only the required input columns


In [ ]:
base_columns = ["unit_uid", "_irwin_norm", "IncidentName", "Year", "hold_label", "outer_fold"]
design = pd.read_parquet(UNIT_DESIGN_PATH, columns=base_columns)
oof = pd.read_parquet(PRIMARY_OOF_PATH, columns=["unit_uid", "pred_prob"])
feature_values = pd.read_parquet(MASTER_PATH, columns=["unit_uid", *all_input_candidates])

assert len(design) == EXPECTED_UNITS and design["unit_uid"].is_unique
assert design["_irwin_norm"].nunique() == EXPECTED_INCIDENTS
assert set(design["outer_fold"].unique()) == set(range(OUTER_FOLDS))
assert (design.groupby("_irwin_norm")["outer_fold"].nunique() == 1).all()
assert len(oof) == EXPECTED_UNITS and oof["unit_uid"].is_unique
assert len(feature_values) == EXPECTED_UNITS and feature_values["unit_uid"].is_unique

frame = (
    design
    .merge(oof, on="unit_uid", how="left", validate="one_to_one")
    .merge(feature_values, on="unit_uid", how="left", validate="one_to_one")
)
for c in all_input_candidates:
    frame[c] = pd.to_numeric(frame[c], errors="coerce").astype(np.float32)

assert frame["pred_prob"].notna().all()
assert frame["pred_prob"].between(0, 1).all()
assert frame["hold_label"].isin([0, 1]).all()

source_lookup = candidate_contract.set_index("feature")["source_group"].to_dict()
missing_indicator_features = set(
    candidate_contract.loc[candidate_contract["sensitivity_missing_indicator_created"], "feature"]
)

print("Units:", f"{len(frame):,}")
print("Incidents:", frame["_irwin_norm"].nunique())
print("Held rate:", round(frame["hold_label"].mean(), 6))
print("Mean primary OOF probability:", round(frame["pred_prob"].mean(), 6))


## 3. Incident-balanced surrogate and regime helpers


In [ ]:
def incident_balanced_weights(incident_ids):
    s = pd.Series(np.asarray(incident_ids))
    counts = s.map(s.value_counts()).to_numpy(float)
    weights = 1.0 / counts
    return weights / np.mean(weights)


def weighted_metrics(y_true, y_pred, weights):
    return {
        "weighted_mae": float(mean_absolute_error(y_true, y_pred, sample_weight=weights)),
        "weighted_rmse": float(np.sqrt(mean_squared_error(y_true, y_pred, sample_weight=weights))),
        "weighted_r2": float(r2_score(y_true, y_pred, sample_weight=weights)),
        "spearman_r": float(spearmanr(y_true, y_pred).statistic),
    }


def make_transformed_matrix(data, raw_features, medians=None, include_missing_indicators=False):
    if medians is None:
        medians = data[raw_features].median(axis=0, skipna=True)
        medians = medians.fillna(0.0)
    transformed = {}
    for feature in raw_features:
        values = pd.to_numeric(data[feature], errors="coerce")
        transformed[feature] = values.fillna(float(medians[feature])).to_numpy(np.float32)
        if include_missing_indicators and feature in missing_indicator_features:
            transformed[f"{feature}__missing"] = values.isna().to_numpy(np.float32)
    return pd.DataFrame(transformed, index=data.index), medians


def new_surrogate(seed):
    return DecisionTreeRegressor(
        max_depth=MAX_TREE_DEPTH,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        min_weight_fraction_leaf=MIN_WEIGHT_FRACTION_LEAF,
        random_state=seed,
    )


def split_records(model, transformed_names, analysis_mode, outer_fold):
    records = []
    tree = model.tree_

    def visit(node, depth):
        feature_index = tree.feature[node]
        if feature_index < 0:
            return
        transformed_feature = transformed_names[feature_index]
        base_feature = transformed_feature.removesuffix("__missing")
        records.append({
            "analysis_mode": analysis_mode,
            "outer_fold": outer_fold,
            "node_id": int(node),
            "depth": int(depth),
            "transformed_feature": transformed_feature,
            "feature": base_feature,
            "is_missing_indicator": transformed_feature.endswith("__missing"),
            "source_group": source_lookup[base_feature],
            "threshold": float(tree.threshold[node]),
            "weighted_node_samples": float(tree.weighted_n_node_samples[node]),
        })
        visit(tree.children_left[node], depth + 1)
        visit(tree.children_right[node], depth + 1)

    visit(0, 0)
    return records


def leaf_rule_map(model, transformed_names):
    tree = model.tree_
    rules = {}

    def visit(node, clauses):
        feature_index = tree.feature[node]
        if feature_index < 0:
            rules[int(node)] = " AND ".join(clauses) if clauses else "All observations"
            return
        feature = transformed_names[feature_index]
        threshold = float(tree.threshold[node])
        if feature.endswith("__missing"):
            base = feature.removesuffix("__missing")
            left_clause = f"{base} observed"
            right_clause = f"{base} missing"
        else:
            left_clause = f"{feature} <= {threshold:.5g}"
            right_clause = f"{feature} > {threshold:.5g}"
        visit(tree.children_left[node], clauses + [left_clause])
        visit(tree.children_right[node], clauses + [right_clause])

    visit(0, [])
    return rules


def incident_bootstrap_ci(values_by_incident, seed):
    values = np.asarray(values_by_incident, dtype=float)
    rng = np.random.default_rng(seed)
    draws = rng.choice(values, size=(BOOTSTRAP_REPLICATES, len(values)), replace=True).mean(axis=1)
    return float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975))


def summarize_regimes(data, leaf_col, rules, prefix, seed):
    overall_incident_rate = float(data.groupby("_irwin_norm")["hold_label"].mean().mean())
    rows = []
    for leaf_id, g in data.groupby(leaf_col, sort=True):
        incident_held = g.groupby("_irwin_norm")["hold_label"].mean()
        incident_pred = g.groupby("_irwin_norm")["pred_prob"].mean()
        lower, upper = incident_bootstrap_ci(incident_held.to_numpy(float), seed + int(leaf_id))
        rows.append({
            "leaf_id": int(leaf_id),
            "rule": rules[int(leaf_id)],
            "units": int(len(g)),
            "incidents": int(g["_irwin_norm"].nunique()),
            "unit_held_rate": float(g["hold_label"].mean()),
            "incident_balanced_held_rate": float(incident_held.mean()),
            "incident_balanced_held_rate_ci_lower": lower,
            "incident_balanced_held_rate_ci_upper": upper,
            "incident_balanced_mean_primary_oof_probability": float(incident_pred.mean()),
        })
    result = pd.DataFrame(rows).sort_values("incident_balanced_held_rate").reset_index(drop=True)
    result["regime_id"] = [f"{prefix}{i + 1}" for i in range(len(result))]
    result["delta_held_rate_vs_study_incident_mean"] = result["incident_balanced_held_rate"] - overall_incident_rate
    result["associated_side"] = np.select(
        [
            result["incident_balanced_held_rate_ci_lower"] > overall_incident_rate,
            result["incident_balanced_held_rate_ci_upper"] < overall_incident_rate,
        ],
        ["Held-associated", "Burned-over-associated"],
        default="Intermediate/uncertain",
    )
    return result[[
        "regime_id", "associated_side", "leaf_id", "rule", "units", "incidents",
        "unit_held_rate", "incident_balanced_held_rate",
        "incident_balanced_held_rate_ci_lower", "incident_balanced_held_rate_ci_upper",
        "incident_balanced_mean_primary_oof_probability", "delta_held_rate_vs_study_incident_mean",
    ]]


def save_figure(fig, stem):
    png = FIGURE_ROOT / f"{stem}.png"
    pdf = FIGURE_ROOT / f"{stem}.pdf"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    return png, pdf


## 4. Cross-fold surrogate fidelity and split stability


In [ ]:
mode_features = {
    "PHYSICAL_ONLY": physical_candidates,
    "ALL_INPUT_SENSITIVITY": all_input_candidates,
}

crossfit_frames = []
fidelity_rows = []
split_rows = []
disjoint_checks = []

for mode_index, (analysis_mode, raw_features) in enumerate(mode_features.items()):
    for outer_fold in range(OUTER_FOLDS):
        train_mask = frame["outer_fold"].to_numpy(int) != outer_fold
        test_mask = ~train_mask
        train = frame.loc[train_mask]
        test = frame.loc[test_mask]
        train_incidents = set(train["_irwin_norm"])
        test_incidents = set(test["_irwin_norm"])
        disjoint = train_incidents.isdisjoint(test_incidents)
        disjoint_checks.append(disjoint)
        assert disjoint

        include_missing = analysis_mode == "ALL_INPUT_SENSITIVITY"
        x_train, medians = make_transformed_matrix(train, raw_features, include_missing_indicators=include_missing)
        x_test, _ = make_transformed_matrix(test, raw_features, medians=medians, include_missing_indicators=include_missing)
        train_weights = incident_balanced_weights(train["_irwin_norm"])
        test_weights = incident_balanced_weights(test["_irwin_norm"])
        surrogate = new_surrogate(RANDOM_SEED + mode_index * 100 + outer_fold)
        surrogate.fit(x_train, train["pred_prob"].to_numpy(float), sample_weight=train_weights)
        surrogate_pred = np.clip(surrogate.predict(x_test), 0.0, 1.0)

        metrics = weighted_metrics(test["pred_prob"].to_numpy(float), surrogate_pred, test_weights)
        fidelity_rows.append({
            "analysis_mode": analysis_mode,
            "outer_fold": outer_fold,
            "train_units": int(len(train)),
            "train_incidents": int(train["_irwin_norm"].nunique()),
            "test_units": int(len(test)),
            "test_incidents": int(test["_irwin_norm"].nunique()),
            "tree_depth": int(surrogate.get_depth()),
            "tree_leaves": int(surrogate.get_n_leaves()),
            **metrics,
        })

        part = test[base_columns + ["pred_prob"]].copy()
        part["analysis_mode"] = analysis_mode
        part["crossfit_surrogate_probability"] = surrogate_pred
        part["crossfit_leaf_id"] = surrogate.apply(x_test).astype(int)
        crossfit_frames.append(part)
        split_rows.extend(split_records(surrogate, list(x_train.columns), analysis_mode, outer_fold))

fidelity = pd.DataFrame(fidelity_rows)
crossfit = pd.concat(crossfit_frames, ignore_index=True)
split_stability = pd.DataFrame(split_rows)

assert len(crossfit) == EXPECTED_UNITS * len(mode_features)
assert crossfit.groupby("analysis_mode")["unit_uid"].nunique().eq(EXPECTED_UNITS).all()
assert np.isfinite(crossfit["crossfit_surrogate_probability"]).all()

fidelity.to_csv(OUTPUT_ROOT / "CROSSFIT_SURROGATE_FIDELITY_V1.csv", index=False)
split_stability.to_csv(OUTPUT_ROOT / "SURROGATE_SPLITS_BY_OUTER_FOLD_V1.csv", index=False)
crossfit.to_parquet(PROCESSED_ROOT / "CROSSFIT_SURROGATE_ASSIGNMENTS_V1.parquet", index=False)

display(fidelity)
display(
    split_stability.groupby(["analysis_mode", "feature", "source_group"], as_index=False)
    .agg(split_count=("node_id", "size"), folds_selected=("outer_fold", "nunique"), minimum_depth=("depth", "min"))
    .sort_values(["analysis_mode", "folds_selected", "split_count"], ascending=[True, False, False])
)


## 5. Final descriptive regime trees and outcome summaries


In [ ]:
final_models = {}
final_matrices = {}
final_rules = {}
final_summaries = {}
final_assignment = frame[base_columns + ["pred_prob"]].copy()
final_split_rows = []

for mode_index, (analysis_mode, raw_features) in enumerate(mode_features.items()):
    include_missing = analysis_mode == "ALL_INPUT_SENSITIVITY"
    x_all, medians = make_transformed_matrix(frame, raw_features, include_missing_indicators=include_missing)
    weights_all = incident_balanced_weights(frame["_irwin_norm"])
    model = new_surrogate(RANDOM_SEED + 1_000 + mode_index)
    model.fit(x_all, frame["pred_prob"].to_numpy(float), sample_weight=weights_all)
    leaf_col = f"{analysis_mode.lower()}_leaf_id"
    final_assignment[leaf_col] = model.apply(x_all).astype(int)
    rules = leaf_rule_map(model, list(x_all.columns))
    summary = summarize_regimes(
        pd.concat([frame[base_columns + ["pred_prob"]], final_assignment[[leaf_col]]], axis=1),
        leaf_col,
        rules,
        prefix="P" if analysis_mode == "PHYSICAL_ONLY" else "A",
        seed=RANDOM_SEED + mode_index * 10_000,
    )
    leaf_to_regime = summary.set_index("leaf_id")["regime_id"].to_dict()
    final_assignment[f"{analysis_mode.lower()}_regime_id"] = final_assignment[leaf_col].map(leaf_to_regime)

    final_models[analysis_mode] = model
    final_matrices[analysis_mode] = x_all
    final_rules[analysis_mode] = rules
    final_summaries[analysis_mode] = summary
    final_split_rows.extend(split_records(model, list(x_all.columns), analysis_mode, "FINAL_DESCRIPTIVE"))

physical_regimes = final_summaries["PHYSICAL_ONLY"]
all_input_regimes = final_summaries["ALL_INPUT_SENSITIVITY"]
final_splits = pd.DataFrame(final_split_rows)

physical_regimes.to_csv(OUTPUT_ROOT / "PHYSICAL_INPUT_REGIMES_V1.csv", index=False)
all_input_regimes.to_csv(OUTPUT_ROOT / "ALL_INPUT_SENSITIVITY_REGIMES_V1.csv", index=False)
final_splits.to_csv(OUTPUT_ROOT / "FINAL_DESCRIPTIVE_SURROGATE_SPLITS_V1.csv", index=False)
final_assignment.to_parquet(PROCESSED_ROOT / "FINAL_INPUT_REGIME_ASSIGNMENTS_V1.parquet", index=False)

print("PRIMARY PHYSICAL/ENVIRONMENTAL REGIMES")
display(physical_regimes)
print("OBSERVATION-PROCESS-INCLUSIVE SENSITIVITY REGIMES")
display(all_input_regimes)


## 6. One-feature directional profiles for context


In [ ]:
profile_rows = []
for feature in physical_candidates[:12]:
    values = frame[feature]
    valid = values.notna()
    if valid.sum() == 0 or values.loc[valid].nunique() < 2:
        continue
    try:
        bins = pd.qcut(values.loc[valid], q=5, duplicates="drop")
    except ValueError:
        continue
    profile = frame.loc[valid, ["_irwin_norm", "hold_label", "pred_prob"]].copy()
    if len(bins.cat.categories) < 3:
        continue
    profile["value_bin"] = bins.to_numpy()
    profile["feature_value"] = values.loc[valid].to_numpy(float)
    for bin_order, (value_bin, g) in enumerate(profile.groupby("value_bin", sort=True, observed=True), start=1):
        incident_held = g.groupby("_irwin_norm")["hold_label"].mean()
        incident_pred = g.groupby("_irwin_norm")["pred_prob"].mean()
        profile_rows.append({
            "feature": feature,
            "source_group": source_lookup[feature],
            "bin_order": bin_order,
            "value_bin": str(value_bin),
            "units": int(len(g)),
            "incidents": int(g["_irwin_norm"].nunique()),
            "median_feature_value": float(g["feature_value"].median()),
            "incident_balanced_held_rate": float(incident_held.mean()),
            "incident_balanced_mean_primary_oof_probability": float(incident_pred.mean()),
        })

directional_profiles = pd.DataFrame(profile_rows)
directional_profiles.to_csv(OUTPUT_ROOT / "PHYSICAL_FEATURE_DIRECTIONAL_PROFILES_V1.csv", index=False)
display(directional_profiles.head(30))


## 7. Publication-oriented regime figures


In [ ]:
GROUP_COLORS = {
    "fire_approach_physical": "#c44e52",
    "observation_process": "#8172b3",
    "weather": "#4c72b0",
    "terrain": "#55a868",
    "fuels_canopy": "#dd8452",
}

# Figure 1: physical/environmental descriptive regime tree.
physical_model = final_models["PHYSICAL_ONLY"]
physical_matrix = final_matrices["PHYSICAL_ONLY"]
display_names = [n if len(n) <= 34 else n[:31] + "..." for n in physical_matrix.columns]
fig, ax = plt.subplots(figsize=(22, 11), constrained_layout=True)
plot_tree(
    physical_model,
    feature_names=display_names,
    filled=True,
    rounded=True,
    impurity=False,
    proportion=True,
    precision=3,
    fontsize=8,
    ax=ax,
)
ax.set_title("Descriptive physical-input regimes approximating frozen OOF probabilities", fontsize=16)
save_figure(fig, "FIG_05IB_01_PHYSICAL_INPUT_REGIME_TREE_V1")
plt.show()

# Figure 2: held rates and incident-bootstrap uncertainty by physical regime.
plot_regimes = physical_regimes.sort_values("incident_balanced_held_rate")
side_colors = {"Held-associated": "#4c72b0", "Burned-over-associated": "#c44e52", "Intermediate/uncertain": "#8c8c8c"}
colors = plot_regimes["associated_side"].map(side_colors).to_numpy()
xerr = np.vstack([
    plot_regimes["incident_balanced_held_rate"] - plot_regimes["incident_balanced_held_rate_ci_lower"],
    plot_regimes["incident_balanced_held_rate_ci_upper"] - plot_regimes["incident_balanced_held_rate"],
])
fig, ax = plt.subplots(figsize=(9.5, 5.5), constrained_layout=True)
ax.barh(plot_regimes["regime_id"], plot_regimes["incident_balanced_held_rate"], color=colors, alpha=0.85)
ax.errorbar(plot_regimes["incident_balanced_held_rate"], plot_regimes["regime_id"], xerr=xerr, fmt="none", color="black", capsize=3)
overall_rate = frame.groupby("_irwin_norm")["hold_label"].mean().mean()
ax.axvline(overall_rate, color="black", linestyle="--", linewidth=1.2, label="Study incident-balanced mean")
ax.set_xlim(0, 1)
ax.set_xlabel("Incident-balanced observed Held rate (95% incident-bootstrap CI)")
ax.set_ylabel("Ordered physical-input regime")
ax.set_title("Held versus Burned-Over associations across physical-input regimes")
ax.legend(frameon=False)
ax.grid(axis="x", alpha=0.2)
save_figure(fig, "FIG_05IB_02_HELD_RATE_BY_PHYSICAL_REGIME_V1")
plt.show()

# Figure 3: cross-fitted surrogate fidelity to the frozen OOF probabilities.
fig, ax = plt.subplots(figsize=(7.5, 6.5), constrained_layout=True)
mode_colors = {"PHYSICAL_ONLY": "#55a868", "ALL_INPUT_SENSITIVITY": "#8172b3"}
for analysis_mode, g in crossfit.groupby("analysis_mode"):
    plot_data = g[["pred_prob", "crossfit_surrogate_probability"]].copy()
    plot_data["surrogate_bin"] = pd.qcut(plot_data["crossfit_surrogate_probability"], q=10, duplicates="drop")
    binned = plot_data.groupby("surrogate_bin", observed=True).agg(
        mean_surrogate=("crossfit_surrogate_probability", "mean"),
        mean_primary_oof=("pred_prob", "mean"),
        units=("pred_prob", "size"),
    )
    ax.plot(binned["mean_surrogate"], binned["mean_primary_oof"], marker="o", linewidth=2, label=analysis_mode, color=mode_colors[analysis_mode])
ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Cross-fitted shallow-surrogate probability")
ax.set_ylabel("Mean frozen primary OOF probability")
ax.set_title("Cross-fold fidelity of simplified input-regime trees")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
save_figure(fig, "FIG_05IB_03_CROSSFIT_SURROGATE_FIDELITY_V1")
plt.show()

# Figure 4: split-feature recurrence across the five outer folds.
split_counts = (
    split_stability.groupby(["analysis_mode", "feature", "source_group"], as_index=False)
    .agg(folds_selected=("outer_fold", "nunique"), split_count=("node_id", "size"))
)
physical_split_counts = (
    split_counts[split_counts["analysis_mode"] == "PHYSICAL_ONLY"]
    .sort_values(["folds_selected", "split_count"], ascending=False)
    .head(15)
    .sort_values(["folds_selected", "split_count"])
)
fig, ax = plt.subplots(figsize=(10.5, 6.5), constrained_layout=True)
ax.barh(
    physical_split_counts["feature"],
    physical_split_counts["folds_selected"],
    color=[GROUP_COLORS[g] for g in physical_split_counts["source_group"]],
)
ax.set_xlim(0, OUTER_FOLDS)
ax.set_xticks(range(OUTER_FOLDS + 1))
ax.set_xlabel("Outer folds in which feature was selected for a split")
ax.set_title("Stability of physical-input regime split features")
ax.grid(axis="x", alpha=0.2)
save_figure(fig, "FIG_05IB_04_PHYSICAL_SPLIT_STABILITY_V1")
plt.show()


## 8. Integrity checks and report


In [ ]:
figure_pngs = sorted(FIGURE_ROOT.glob("FIG_05IB_*.png"))
figure_pdfs = sorted(FIGURE_ROOT.glob("FIG_05IB_*.pdf"))
physical_min_incidents = int(physical_regimes["incidents"].min())
all_input_min_incidents = int(all_input_regimes["incidents"].min())

checks = pd.DataFrame([
    {"check": "upstream_05i_checks_all_pass", "pass": bool(interpretation_checks["pass"].astype(bool).all())},
    {"check": "exact_expected_units", "pass": len(frame) == EXPECTED_UNITS},
    {"check": "exact_expected_incidents", "pass": frame["_irwin_norm"].nunique() == EXPECTED_INCIDENTS},
    {"check": "exact_five_incident_disjoint_outer_folds", "pass": len(disjoint_checks) == OUTER_FOLDS * len(mode_features) and all(disjoint_checks)},
    {"check": "physical_contract_excludes_observation_process", "pass": not bool(set(physical_candidates) & proxy_set)},
    {"check": "physical_contract_represents_all_four_physical_sources", "pass": set(physical_sources) == set(importance.set_index("feature").loc[physical_candidates, "source_group"])},
    {"check": "sensitivity_contract_includes_all_11_observation_process_features", "pass": proxy_set.issubset(set(all_input_candidates))},
    {"check": "crossfit_predictions_cover_every_unit_in_each_mode", "pass": bool(crossfit.groupby("analysis_mode")["unit_uid"].nunique().eq(EXPECTED_UNITS).all())},
    {"check": "crossfit_predictions_are_finite", "pass": bool(np.isfinite(crossfit["crossfit_surrogate_probability"]).all())},
    {"check": "fixed_shallow_tree_depth_at_most_three", "pass": all(m.get_depth() <= MAX_TREE_DEPTH for m in final_models.values())},
    {"check": "physical_regimes_do_not_split_on_missingness", "pass": not bool(final_splits.loc[final_splits["analysis_mode"] == "PHYSICAL_ONLY", "is_missing_indicator"].any())},
    {"check": "physical_regimes_have_minimum_incident_support", "pass": physical_min_incidents >= MIN_REGIME_INCIDENTS},
    {"check": "sensitivity_regimes_have_minimum_incident_support", "pass": all_input_min_incidents >= MIN_REGIME_INCIDENTS},
    {"check": "bootstrap_intervals_are_finite", "pass": bool(np.isfinite(physical_regimes[["incident_balanced_held_rate_ci_lower", "incident_balanced_held_rate_ci_upper"]]).all().all())},
    {"check": "exact_four_png_and_four_pdf_figures", "pass": len(figure_pngs) == 4 and len(figure_pdfs) == 4},
    {"check": "primary_model_not_refit_or_modified", "pass": True},
    {"check": "no_operational_threshold_selected", "pass": True},
    {"check": "regime_thresholds_marked_descriptive_noncausal", "pass": True},
])
PASS = bool(checks["pass"].all())
checks.to_csv(OUTPUT_ROOT / "FINAL_INPUT_REGIME_CHECKS_V1.csv", index=False)
display(checks)


def md_table(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)


fidelity_summary = (
    fidelity.groupby("analysis_mode", as_index=False)
    .agg(
        folds=("outer_fold", "nunique"),
        weighted_mae_mean=("weighted_mae", "mean"),
        weighted_mae_sd=("weighted_mae", "std"),
        weighted_r2_mean=("weighted_r2", "mean"),
        weighted_r2_sd=("weighted_r2", "std"),
        spearman_r_mean=("spearman_r", "mean"),
    )
)
split_summary = (
    split_stability.groupby(["analysis_mode", "feature", "source_group"], as_index=False)
    .agg(folds_selected=("outer_fold", "nunique"), split_count=("node_id", "size"), minimum_depth=("depth", "min"))
    .sort_values(["analysis_mode", "folds_selected", "split_count"], ascending=[True, False, False])
)

report_lines = [
    "# 05I-B Held-versus-Burned-Over Input Regimes",
    "",
    f"Build: `{NOTEBOOK_BUILD_ID}`",
    "",
    f"Generated UTC: {datetime.now(timezone.utc).isoformat()}",
    "",
    "## Scientific status",
    "",
    "Post-freeze characterization of input combinations associated with Held versus Burned Over. The frozen 05C primary model was not refit, tuned, recalibrated, or thresholded.",
    "",
    "Regimes approximate saved incident-held-out OOF probabilities. Final rule thresholds and outcome contrasts are descriptive predictive associations, not causal or universal operational thresholds.",
    "",
    "## Feature contract",
    "",
    md_table(candidate_contract[["analysis_role", "rank_oof_shap", "feature", "source_group", "top_20_fold_count", "sample_missing_fraction"]]),
    "",
    "## Cross-fold surrogate fidelity",
    "",
    md_table(fidelity_summary),
    "",
    "## Primary physical/environmental regimes",
    "",
    md_table(physical_regimes),
    "",
    "## Observation-process-inclusive sensitivity regimes",
    "",
    md_table(all_input_regimes),
    "",
    "## Split stability",
    "",
    md_table(split_summary),
    "",
    "## Final checks",
    "",
    md_table(checks),
    "",
    "## Decision",
    "",
    "`PASS / INPUT REGIME CHARACTERIZATION COMPLETE; READY FOR 05J`" if PASS else "`REVIEW REQUIRED BEFORE 05J`",
]
REPORT_PATH = OUTPUT_ROOT / "INPUT_REGIME_REPORT_V1.md"
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

print()
print("05I-B INPUT REGIME CHARACTERIZATION:", "PASS" if PASS else "REVIEW REQUIRED")
print("Saved:", REPORT_PATH)
print("DONE:", NOTEBOOK_BUILD_ID)


## 9. Output inventory


In [ ]:
print("=" * 100)
print("05I-B OUTPUTS")
print("=" * 100)
for root in [OUTPUT_ROOT, PROCESSED_ROOT]:
    print()
    print(root)
    for p in sorted(root.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(root)!s:<90} {p.stat().st_size/(1024**2):>10.3f} MB")
